# P01 (basic) — Punktwolken-Grundlagen: Nachbarschaften, Downsampling & Normalen

**Modul 20 — 3D Point Cloud Processing**

Bevor man Punktwolken registrieren oder segmentieren kann, braucht man drei Grundoperationen:
1. **Nachbarschaften** effizient finden (kNN / Radius) — per **kd-Baum**.
2. **Downsampling** (Voxel-Grid) — Punktzahl runter, Dichte gleichmaessig.
3. **Normalen schaetzen** — per **lokaler PCA** (Eigenzerlegung der Kovarianzmatrix).

Du baust alle drei. Der Clou fuer die Normalen: Wir sampeln eine **Kugel**, deren wahre Normalen
wir kennen (sie zeigen radial nach aussen) — so kannst du deine Schaetzung **gegen die ground truth
pruefen**.

### Ziel
- kd-Baum-Nachbarschaften mit `scipy.spatial.cKDTree` nutzen,
- **Voxel-Downsampling** implementieren,
- **Normalen + Kruemmung** ueber die lokale Kovarianz-PCA schaetzen (Kap. 5 des Skripts),
- die Schaetzung gegen die analytischen Kugelnormalen validieren.

### Format
Jupyter-Notebook — Punktwolken lebt von 3D-Visualisierung neben der Rechnung.

### Vorwissen
PCA / Eigenzerlegung (Modul 05), Kap. 3-5 des Modul-20-Skripts.

### Aufgaben
Die meisten Zellen sind vorgegeben; an den `# TODO`-Stellen implementierst du Downsampling und
Normalen-Schaetzung. Loesung in `solution/`.


## Setup
`numpy`, `scipy` (kd-Baum), `matplotlib` (3D) — alle in der `.venv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.spatial import cKDTree
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)


## Teil A — Eine synthetische Punktwolke (Kugel mit Rauschen)

Wir sampeln Punkte gleichmaessig auf einer Kugel (Zentrum `c`, Radius `R`) und addieren
radiales Gauss-Rauschen. Die **wahre Normale** in einem Punkt ist die radiale Richtung
$(\mathbf p - \mathbf c)/\lVert\mathbf p - \mathbf c\rVert$ — die heben wir als ground truth auf.
(Zelle vorgegeben.)

In [ ]:
def sample_sphere(n, center, R, noise=0.01, rng=rng):
    # gleichverteilte Richtungen: normierte Gauss-Vektoren
    dirs = rng.normal(size=(n, 3))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
    radii = R + rng.normal(0, noise, n)          # radiales Rauschen
    pts = center + dirs * radii[:, None]
    gt_normals = dirs                            # wahre (aeussere) Normale = Richtung
    return pts, gt_normals

center = np.array([0.0, 0.0, 0.0]); R = 1.0
pts, gt_normals = sample_sphere(2500, center, R, noise=0.01)
print("Punktwolke:", pts.shape)

fig = plt.figure(figsize=(6, 6)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=3, alpha=0.4)
ax.set_title("Kugel-Punktwolke (2500 Punkte)")
try: ax.set_box_aspect((1, 1, 1))
except Exception: pass
plt.show()


## Teil B — Nachbarschaften mit dem kd-Baum

Der **kd-Baum** beantwortet kNN- und Radius-Anfragen im Mittel in $O(\log n)$ statt $O(n)$.
`scipy.spatial.cKDTree` baut ihn in $O(n\log n)$. (Zelle vorgegeben — beobachte die Ausgaben.)

In [ ]:
tree = cKDTree(pts)

# kNN: die 8 naechsten Nachbarn zum ersten Punkt
dist, idx = tree.query(pts[0], k=8)
print("8-NN von Punkt 0: Indizes", idx)
print("Distanzen        ", dist.round(3))

# Radius-Suche: alle Punkte im Radius 0.1 um Punkt 0
nb = tree.query_ball_point(pts[0], r=0.1)
print(f"Punkte im Radius 0.1 um Punkt 0: {len(nb)}")


## Teil C — Voxel-Downsampling

Lege ein Gitter der Kantenlaenge `voxel` ueber die Wolke und **ersetze alle Punkte in einem Voxel
durch ihren Schwerpunkt**. Ergebnis: hoechstens ein Punkt je besetztem Voxel, gleichmaessige Dichte.

**Deine Aufgabe:** Implementiere `voxel_downsample`. Schritte:
- Voxel-Index jedes Punktes: `np.floor((pts - pts.min(0)) / voxel)` als ganzzahliges Tripel.
- Punkte mit gleichem Voxel-Index gruppieren, je Gruppe den **Mittelwert** bilden.
(Tipp: `np.unique(..., axis=0, return_inverse=True)` liefert Gruppenlabels.)

In [ ]:
def voxel_downsample(pts, voxel):
    keys = np.floor((pts - pts.min(0)) / voxel).astype(np.int64)   # (n,3) Voxel-Indizes
    # TODO: gruppiere nach eindeutigem Voxel-Schluessel und mittle die Punkte je Gruppe
    # uniq, inv = np.unique(keys, axis=0, return_inverse=True)
    # summe je Gruppe / anzahl je Gruppe
    ...  # TODO
    return down  # (m,3) mit m <= n

down = voxel_downsample(pts, voxel=0.15)
print(f"vorher {len(pts)} Punkte -> nachher {len(down)} Punkte (Voxel 0.15)")


**Erwartung.** Von 2500 Punkten bleiben je nach Voxelgroesse einige Hundert (bei 0.15 grob
~250-350). Die Kugelform bleibt erhalten, die Dichte wird gleichmaessiger.

## Teil D — Normalen-Schaetzung via lokaler PCA

Der Kern (Skript Kap. 5). Fuer jeden Punkt:
1. die $k$ naechsten Nachbarn holen (kd-Baum),
2. ihre **Kovarianzmatrix** $\mathbf C = \frac{1}{k}\sum (\mathbf q-\bar{\mathbf q})(\mathbf q-\bar{\mathbf q})^\top$,
3. Eigenzerlegung; die **Normale = Eigenvektor zum kleinsten Eigenwert** (Richtung minimaler Varianz),
4. **Kruemmung** $\sigma = \lambda_0/(\lambda_0+\lambda_1+\lambda_2)$.

Das Vorzeichen orientieren wir nach **aussen** (weg vom Zentrum): ist $\mathbf n\cdot(\mathbf p-\mathbf c)<0$,
drehe $\mathbf n$ um.

**Deine Aufgabe:** Fuelle Kovarianz, Normale und Kruemmung ein.

In [ ]:
def estimate_normals(pts, k=16, viewpoint_outward_center=None):
    tree = cKDTree(pts)
    normals = np.zeros_like(pts)
    curvature = np.zeros(len(pts))
    for i, p in enumerate(pts):
        _, idx = tree.query(p, k=k)
        nb = pts[idx]
        nb_c = nb - nb.mean(0)                     # zentrieren
        # TODO: Kovarianzmatrix C (3x3)
        C = ...  # TODO
        evals, evecs = np.linalg.eigh(C)           # eigh: aufsteigende Eigenwerte
        # TODO: Normale = Eigenvektor zum KLEINSTEN Eigenwert (Spalte 0)
        n = ...  # TODO
        # TODO: Kruemmung = kleinster Eigenwert / Summe der Eigenwerte
        curv = ...  # TODO
        # Orientierung nach aussen (weg vom Zentrum), falls Zentrum gegeben
        if viewpoint_outward_center is not None:
            if np.dot(n, p - viewpoint_outward_center) < 0:
                n = -n
        normals[i] = n; curvature[i] = curv
    return normals, curvature

est_normals, curv = estimate_normals(pts, k=16, viewpoint_outward_center=center)

# Validierung gegen die wahren Kugelnormalen: Winkelfehler
cos = np.clip(np.sum(est_normals * gt_normals, axis=1), -1, 1)
ang_err_deg = np.rad2deg(np.arccos(cos))
print(f"Normalen-Winkelfehler: Median {np.median(ang_err_deg):.2f} deg, "
      f"Mittel {ang_err_deg.mean():.2f} deg")
print(f"Kruemmung (Kugel ~ konstant, gering): Mittel {curv.mean():.4f}")


**Erwartung / Selbstcheck.** Der **Median-Winkelfehler** der geschaetzten Normalen sollte klein
sein (grob **1-5 Grad** bei k=16 und diesem Rauschen) — deine lokale PCA rekonstruiert die
Kugelnormalen also gut. Die **Kruemmung** ist klein und ueber die Kugel etwa konstant (glatte
Flaeche); an einer Kante/Ecke waere sie gross. Erhoehe das Rauschen oder verkleinere `k`, und der
Fehler steigt — Normalen-Schaetzung ist ein Bias-Varianz-Tradeoff in der Nachbarschaftsgroesse.

## Teil E — Normalen visualisieren

(Vorgegeben.) Wir zeichnen eine Teilmenge der Punkte mit ihren geschaetzten Normalen als Pfeile —
sie sollten alle schoen radial nach aussen zeigen.

In [ ]:
sel = rng.choice(len(pts), 200, replace=False)
fig = plt.figure(figsize=(7, 7)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, alpha=0.15, color="gray")
ax.quiver(pts[sel, 0], pts[sel, 1], pts[sel, 2],
          est_normals[sel, 0], est_normals[sel, 1], est_normals[sel, 2],
          length=0.25, color="crimson", linewidth=0.7)
ax.set_title("Geschaetzte Normalen (zeigen radial nach aussen)")
try: ax.set_box_aspect((1, 1, 1))
except Exception: pass
plt.show()


## Fazit

Du hast die drei Grundoperationen jeder Punktwolken-Pipeline gebaut:
- **kd-Baum**-Nachbarschaften (der Motor unter allem Folgenden),
- **Voxel-Downsampling** (Vorverarbeitung),
- **Normalen + Kruemmung via lokaler PCA** — validiert gegen die Kugel-ground-truth.

Diese Bausteine sind die Voraussetzung fuer die naechsten Projekte: **ICP-Registrierung**
(medium) braucht Nachbarschaften fuer die Korrespondenzsuche, und die **Segmentierungspipeline**
(final) braucht Normalen und Nachbarschaften fuer RANSAC + Clustering.